In [1]:
import os
import pickle
import pandas as pd

from getpass import getpass
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
import time

In [ ]:
eval_df = pd.read_csv("../data/rag_generation_results_50.csv")

with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

documents_by_id = {
    doc.metadata["doc_id"]: doc
    for doc in documents
}

print(
    eval_df.iloc[0]["retrieved_doc_ids"]
)

print(
    documents[0].page_content
)

print(
    documents[0].metadata
)

1155.KL_817315_2|1155.KL_403189_2|1155.KL_733739_1|1155.KL_796934_5|1155.KL_395620_1
PETRONAS Chemicals Group Bhd (KL: PCHEM ) rose to a three-month high amid a rally in petrochemical refiners as oil prices surged amid renewed conflict in the Middle East. Brent, the global benchmark for crude oil, extended its gains above US$100 while West Texas Intermediate, which guides US markets, rose to US$103 per barrel, boosting hopes for higher petrochemical prices and spreads. — PETRONAS Chemicals rises to three-month high, refiners climb as Brent tops US$100
{'doc_id': '5183.KL_818088_6', 'text_hash': '0d1e59927fceff3c20543d872c0ef9e524d2339e', 'company': 'Petronas Chemicals', 'ticker': '5183.KL', 'nid': '818088', 'info_index': 6, 'date': '2026-09-14', 'article_title': 'Kelington, NuEnergy, GFM, Exsim Hospitality, Kerjaya Prospek, PETRONAS Chemicals, Tan Chong Motor and United Asiapac', 'article_url': 'https://theedgemalaysia.com/node/818088', 'sentiment': 'positive', 'sentiment_score': 0.920

In [3]:
def build_retrieved_context(retrieved_doc_ids):

    doc_ids = str(
        retrieved_doc_ids
    ).split("|")

    context_parts = []

    for i, doc_id in enumerate(
        doc_ids,
        start=1
    ):

        doc_id = doc_id.strip()

        if doc_id not in documents_by_id:
            continue

        doc = documents_by_id[
            doc_id
        ]

        context_parts.append(
            f"""
[Source {i}]

Company: {doc.metadata["company"]}
Date: {doc.metadata["date"]}

Content:
{doc.page_content}
""".strip()
        )

    return "\n\n".join(
        context_parts
    )

In [4]:
context = build_retrieved_context(
    eval_df.iloc[0]["retrieved_doc_ids"]
)

print(context)

[Source 1]

Company: Maybank
Date: 2026-09-08

Content:
Malayan Banking Bhd (KL: MAYBANK ) has received Bank Negara Malaysia approval to acquire the remaining 30.95% stake in Maybank Ageas Holdings Bhd, the holding company of Etiqa, from Belgian insurer Ageas SA for RM4.83 billion. Following the approval on Sept 1, Maybank has entered into an unconditional share purchase agreement with Ageas. The acquisition will be funded entirely through Maybank’s internally generated funds. — Maybank gets BNM nod for RM4.83b buyout of remaining stake in Etiqa’s holding company

[Source 2]

Company: Maybank
Date: 2018-04-20

Content:
Malayan Banking Bhd (Maybank) has obtained approval from Bank Negara Malaysia to acquire Permodalan Nasional Bhd’s fund management firms, namely, Amanah Mutual Bhd (AMB) and Singapore Unit Trusts Ltd (SUTL), for RM51 million cash.

[Source 3]

Company: Maybank
Date: 2024-11-13

Content:
Malayan Banking Bhd (KL: MAYBANK ) is considering options including buying out Ageas 

In [5]:
class RAGEvaluation(BaseModel):

    correctness: float = Field(
        ge=0.0,
        le=1.0
    )

    answer_relevance: float = Field(
        ge=0.0,
        le=1.0
    )

    faithfulness: float = Field(
        ge=0.0,
        le=1.0
    )

    explanation: str

In [11]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "OpenAI API Key: "
    )

judge_llm = ChatOpenAI(
    model="gpt-6-luna",
)

structured_judge = (
    judge_llm.with_structured_output(
        RAGEvaluation
    )
)

In [7]:
evaluation_prompt = """
You are evaluating a Retrieval-Augmented Generation
system for Malaysian financial news.

Evaluate the GENERATED ANSWER using the following metrics.


CORRECTNESS:

Compare the generated answer with the expected answer.

1.0 means the required facts are correct and complete.

0.0 means the answer is incorrect or contradicts
the expected answer.

Partial scores are allowed.


ANSWER RELEVANCE:

Evaluate whether the generated answer directly
answers the user's question.

1.0 means it directly and concisely answers
the question.

0.0 means it does not answer the question.


FAITHFULNESS:

Evaluate whether the factual claims in the generated
answer are supported by the retrieved context.

1.0 means all factual claims are supported.

0.0 means important claims are invented,
unsupported or contradict the context.


QUESTION:
{question}


EXPECTED ANSWER:
{expected_answer}


GENERATED ANSWER:
{generated_answer}


RETRIEVED CONTEXT:
{context}
"""

In [8]:
def evaluate_one_answer(row):

    context = build_retrieved_context(
        row["retrieved_doc_ids"]
    )

    prompt = evaluation_prompt.format(
        question=row["question"],
        expected_answer=row["expected_answer"],
        generated_answer=row["generated_answer"],
        context=context
    )

    evaluation = structured_judge.invoke(
        prompt
    )

    return evaluation

In [12]:
rag_evaluation_results = []


for i, row in eval_df.iterrows():

    print(
        f"Evaluation {i + 1}/{len(eval_df)}"
    )

    try:

        evaluation = evaluate_one_answer(
            row
        )

        rag_evaluation_results.append(
            {
                "question_id":
                    row["question_id"],

                "question":
                    row["question"],

                "expected_answer":
                    row["expected_answer"],

                "generated_answer":
                    row["generated_answer"],

                "correctness":
                    evaluation.correctness,

                "answer_relevance":
                    evaluation.answer_relevance,

                "faithfulness":
                    evaluation.faithfulness,

                "explanation":
                    evaluation.explanation
            }
        )
        time.sleep(17)


    except Exception as e:

        print(
            "Erreur",
            row["question_id"],
            e
        )


Evaluation 1/50
Evaluation 2/50
Evaluation 3/50
Evaluation 4/50
Evaluation 5/50
Evaluation 6/50
Evaluation 7/50
Evaluation 8/50
Evaluation 9/50
Evaluation 10/50
Evaluation 11/50
Evaluation 12/50
Evaluation 13/50
Evaluation 14/50
Evaluation 15/50
Evaluation 16/50
Evaluation 17/50
Evaluation 18/50
Evaluation 19/50
Evaluation 20/50
Evaluation 21/50
Evaluation 22/50
Evaluation 23/50
Evaluation 24/50
Evaluation 25/50
Evaluation 26/50
Evaluation 27/50
Evaluation 28/50
Evaluation 29/50
Evaluation 30/50
Evaluation 31/50
Evaluation 32/50
Evaluation 33/50
Evaluation 34/50
Evaluation 35/50
Evaluation 36/50
Evaluation 37/50
Evaluation 38/50
Evaluation 39/50
Evaluation 40/50
Evaluation 41/50
Evaluation 42/50
Evaluation 43/50
Evaluation 44/50
Evaluation 45/50
Evaluation 46/50
Evaluation 47/50
Evaluation 48/50
Evaluation 49/50
Evaluation 50/50


In [13]:
rag_eval_df = pd.DataFrame(
    rag_evaluation_results
)


print(
    "Nombre de réponses évaluées :",
    len(rag_eval_df)
)


display(
    rag_eval_df.head()
)

Nombre de réponses évaluées : 50


,question_id,question,expected_answer,generated_answer,correctness,answer_relevance,faithfulness,explanation
0,Q01,What approval did Maybank receive in September...,Bank Negara Malaysia approved Maybank's acquis...,"In September 2026, Maybank received Bank Negar...",1.0,1.0,1.0,The answer correctly states that Bank Negara M...
1,Q02,What role was Maybank Indonesia approved to ta...,PT Bank Maybank Indonesia was approved to serv...,"In June 2026, PT Bank Maybank Indonesia Tbk se...",1.0,1.0,1.0,The answer directly identifies Maybank Indones...
2,Q03,What did Maybank clarify in June 2026 about th...,Maybank said Maybank Indonesia was not under i...,"In June 2026, Maybank clarified that its Indon...",1.0,1.0,1.0,The answer directly states that Maybank Indone...
3,Q04,How did Maybank's net profit perform in 1QFY20...,Net profit fell 4% year on year to RM2.48 bill...,The provided documents do not contain informat...,0.0,0.2,1.0,The answer omits the expected result: net prof...
4,Q05,How much subordinated sukuk did Maybank issue ...,Maybank issued RM1.2 billion of subordinated s...,Maybank issued RM1.2 billion in subordinated s...,1.0,1.0,1.0,The answer correctly states the RM1.2 billion ...


In [14]:
summary = pd.DataFrame(
    {
        "Metric": [
            "Correctness",
            "Answer Relevance",
            "Faithfulness"
        ],

        "Score": [
            rag_eval_df[
                "correctness"
            ].mean(),

            rag_eval_df[
                "answer_relevance"
            ].mean(),

            rag_eval_df[
                "faithfulness"
            ].mean()
        ]
    }
)


display(summary)

,Metric,Score
0,Correctness,0.813
1,Answer Relevance,0.860
2,Faithfulness,1.000


In [16]:
print(
    "Perfect correctness:",
    (rag_eval_df["correctness"] == 1).sum(),
    "/",
    len(rag_eval_df)
)

print(
    "Perfect relevance:",
    (rag_eval_df["answer_relevance"] == 1).sum(),
    "/",
    len(rag_eval_df)
)

print(
    "Perfect faithfulness:",
    (rag_eval_df["faithfulness"] == 1).sum(),
    "/",
    len(rag_eval_df)
)


failed_questions = rag_eval_df[
    rag_eval_df["correctness"] == 0
]

display(
    failed_questions[
        [
            "question_id",
            "question",
            "expected_answer",
            "generated_answer",
            "correctness",
            "answer_relevance",
            "faithfulness",
            "explanation"
        ]
    ]
)

Perfect correctness: 30 / 50
Perfect relevance: 40 / 50
Perfect faithfulness: 50 / 50


,question_id,question,expected_answer,generated_answer,correctness,answer_relevance,faithfulness,explanation
3,Q04,How did Maybank's net profit perform in 1QFY20...,Net profit fell 4% year on year to RM2.48 bill...,The provided documents do not contain informat...,0.0,0.2,1.0,The answer omits the expected result: net prof...
8,Q09,How did Maybank's net profit change in 3QFY2025?,Maybank's net profit rose 3% year on year to R...,The provided documents do not contain informat...,0.0,0.2,1.0,The generated answer does not give the expecte...
10,Q11,Why did CIMB Group's 1QFY2026 net profit decli...,Net profit slipped to RM1.916 billion from RM1...,The provided documents do not contain informat...,0.0,0.0,1.0,The generated answer gives neither the reason ...
21,Q22,What net profit did Public Bank report for 1QF...,Public Bank reported RM1.75 billion of net pro...,The provided documents do not contain informat...,0.0,0.2,1.0,The generated answer does not provide the expe...
37,Q38,What were PETRONAS Chemicals' 2QFY2026 net pro...,PETRONAS Chemicals returned to profit with RM4...,The retrieved documents do not contain enough ...,0.0,0.1,1.0,The answer omits all three expected 2QFY2026 f...
45,Q46,Compare Maybank's profit performance in 4QFY20...,In 4QFY2025 Maybank's net profit rose 5.7% yea...,The provided documents do not contain informat...,0.0,0.2,1.0,The answer omits and fails to provide the expe...


In [15]:
rag_eval_df.to_csv("../data/rag_generation_evaluation.csv",index=False)